# 🐛 Bug Prediction System — v2 (SZZ-Aligned)

**Updated pipeline** that ingests real commit data from `szz_training_ready.csv`.

All fixes applied vs v1:
```
Fix 1 → Loads real SZZ CSV instead of synthetic generate_dataset()
Fix 2 → Feature columns match szz_v3_stable.ipynb output exactly
Fix 3 → Temporal train/test split (by committed_at) — no future leakage
Fix 4 → prior_bugs_author recomputed inside CV folds only
Fix 5 → commit_type and commit_message excluded from features
Fix 6 → Confidence-weighted sample weights passed to models
Fix 7 → SMOTE oversampling on training fold only
Fix 8 → TimeSeriesSplit cross-validation instead of random KFold
```

**Steps:**
1. Install dependencies
2. Import libraries
3. Load SZZ dataset & validate
4. Exploratory Data Analysis
5. Temporal train/test split
6. Feature engineering (fold-safe)
7. Preprocessing pipeline
8. Train models with TimeSeriesSplit CV
9. Evaluate on held-out test set
10. Save & reload model
11. Predict on new commit

## Step 1 — Install dependencies

In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib imbalanced-learn --quiet
print('✅ Done')

## Step 2 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_recall_curve, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ All libraries imported successfully!')

## Step 3 — Load SZZ dataset & validate

Loads `szz_training_ready.csv` produced by `szz_v3_stable.ipynb`.
Performs column validation, confidence filtering, and a sanity report.

In [ ]:
# ── Load the full CSV (with metadata columns still present) ──────────────────
# We load szz_final_labels.csv (the full version) so we keep 'committed_at'
# for temporal splitting. szz_training_ready.csv already drops it.
CSV_PATH = 'szz_final_labels.csv'   # ← change to 'szz_training_ready.csv' if you only have that

df_raw = pd.read_csv(CSV_PATH)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns  : {list(df_raw.columns)}')

# ── Columns produced by szz_v3_stable.ipynb ─────────────────────────────────
# These are the FEATURE columns we will actually train on.
# Metadata columns (sha, message, author, etc.) are intentionally excluded
# to prevent label leakage via commit message keywords.
EXPECTED_FEATURE_COLS = [
    'lines_added', 'lines_deleted', 'files_changed',
    'test_files_changed', 'num_methods', 'avg_complexity',
    'commit_hour', 'day_of_week', 'is_weekend', 'is_night_commit',
    'churn_ratio', 'test_ratio', 'complexity_per_file',
    'prior_bugs_author',
]

# Columns to keep for splitting / weighting but NOT for training
META_COLS = [
    'commit_sha_full', 'commit_sha', 'repo', 'committed_at',
    'commit_message', 'author', 'language', 'commit_type',
    'language_group', 'time_period', 'linked_issue',
    'label_source', 'confidence',
]

TARGET = 'is_buggy'

# ── Validate columns ─────────────────────────────────────────────────────────
missing_feat = [c for c in EXPECTED_FEATURE_COLS if c not in df_raw.columns]
missing_meta = [c for c in META_COLS            if c not in df_raw.columns]

if missing_feat:
    print(f'⚠️  Missing feature columns: {missing_feat}')
    print('   → These will be zero-filled. Check your SZZ notebook output.')
else:
    print('✅ All expected feature columns present')

if missing_meta:
    print(f'ℹ️  Missing meta columns (non-critical): {missing_meta}')

# Zero-fill any missing feature columns so training still works
for col in EXPECTED_FEATURE_COLS:
    if col not in df_raw.columns:
        df_raw[col] = 0

# If committed_at is missing, create a synthetic sequential order
if 'committed_at' not in df_raw.columns:
    print('⚠️  committed_at not found — using row order for temporal split')
    df_raw['committed_at'] = pd.date_range('2018-01-01', periods=len(df_raw), freq='h')

# confidence defaults to 1.0 if not present
if 'confidence' not in df_raw.columns:
    df_raw['confidence'] = 1.0

# ── Filter: keep only sufficiently confident labels ──────────────────────────
CONFIDENCE_THRESHOLD = 0.3
df = df_raw[df_raw['confidence'] >= CONFIDENCE_THRESHOLD].copy()
print(f'\nAfter confidence filter (>= {CONFIDENCE_THRESHOLD}): {len(df)} rows  ({len(df_raw)-len(df)} dropped)')

# ── Label sanity report ──────────────────────────────────────────────────────
print(f'\n=== Label Report ===')
print(f'Total commits : {len(df)}')
print(f'Bug rate      : {df[TARGET].mean():.1%}')
print(f'\nBy label_source:')
if 'label_source' in df.columns:
    print(df.groupby('label_source')[TARGET].agg(['count','sum','mean']).rename(columns={'count':'n','sum':'bugs','mean':'bug_rate'}))

if len(df) < 500:
    print(f'\n⚠️  Only {len(df)} samples. Consider increasing COMMITS_PER_REPO and MAX_ISSUES in SZZ notebook.')
if df[TARGET].mean() < 0.05:
    print('⚠️  Bug rate < 5% — very imbalanced. SMOTE will help but consider adding more repos.')
if df[TARGET].mean() > 0.5:
    print('⚠️  Bug rate > 50% — keyword fallback may have over-labeled. Review label_source breakdown.')

df.head(3)

## Step 4 — Exploratory Data Analysis

In [ ]:
# 4a. Basic stats & missing values
print('=== Feature Stats ===')
display(df[EXPECTED_FEATURE_COLS + [TARGET]].describe().T.round(2))

print('\n=== Missing Values in Feature Columns ===')
mv = df[EXPECTED_FEATURE_COLS].isnull().sum()
print(mv[mv > 0] if mv.any() else '  None ✅')

In [ ]:
# 4b. Class distribution + correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df[TARGET].value_counts()
axes[0].pie(counts, labels=['Clean', 'Buggy'], autopct='%1.1f%%',
            colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Class Distribution')

corr_cols = EXPECTED_FEATURE_COLS + [TARGET]
corr = df[corr_cols].corr()
sns.heatmap(
    corr[['is_buggy']].sort_values('is_buggy', ascending=False),
    ax=axes[1], annot=True, fmt='.2f', cmap='RdYlGn_r', vmin=-1, vmax=1
)
axes[1].set_title('Feature Correlation with Bug Label')

plt.tight_layout()
plt.show()

In [ ]:
# 4c. Key feature distributions: clean vs buggy
plot_features = ['lines_added', 'files_changed', 'avg_complexity',
                 'churn_ratio', 'test_ratio', 'prior_bugs_author']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    if feat not in df.columns:
        continue
    # Cap at 99th percentile so outliers don't squash the plot
    cap = df[feat].quantile(0.99)
    subset = df[df[feat] <= cap]
    subset.groupby(TARGET)[feat].plot.kde(ax=axes[i], legend=True)
    axes[i].set_title(feat.replace('_', ' ').title())
    axes[i].legend(['Clean', 'Buggy'])
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions: Clean vs Buggy Commits', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 4d. Bug rate over time (temporal trend)
df['committed_at'] = pd.to_datetime(df['committed_at'], utc=True, errors='coerce')
df = df.dropna(subset=['committed_at'])
df = df.sort_values('committed_at').reset_index(drop=True)

df['year_month'] = df['committed_at'].dt.to_period('M')
monthly = df.groupby('year_month')[TARGET].mean()

plt.figure(figsize=(12, 4))
monthly.plot(color='#E53935', linewidth=2)
plt.axhline(df[TARGET].mean(), linestyle='--', color='gray', label=f'Overall mean: {df[TARGET].mean():.1%}')
plt.title('Bug Rate Over Time (Monthly)')
plt.ylabel('Bug Rate')
plt.xlabel('')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Date range: {df["committed_at"].min().date()} → {df["committed_at"].max().date()}')
print(f'Total months: {len(monthly)}')

## Step 5 — Temporal Train / Test Split

**Why temporal?** A random split would let the model train on commits from 2023
and test on commits from 2019 — simulating knowledge of the future.
Real deployment always predicts *future* commits, so we split by time:
- **Train**: earliest 80% of commits (chronologically)
- **Test**: latest 20% of commits

In [ ]:
# df is already sorted by committed_at from Step 4d
SPLIT_RATIO = 0.80
split_idx   = int(len(df) * SPLIT_RATIO)
split_date  = df.iloc[split_idx]['committed_at']

df_train = df.iloc[:split_idx].copy()
df_test  = df.iloc[split_idx:].copy()

print(f'Split date  : {split_date.date()}')
print(f'Train size  : {len(df_train)}  ({df_train[TARGET].mean():.1%} bug rate)')
print(f'Test  size  : {len(df_test)}   ({df_test[TARGET].mean():.1%} bug rate)')

# ── Recompute prior_bugs_author on training set only ────────────────────────
# This is the fold-safe version: rolling sum is computed separately for
# train and test so the test set never sees future label information.
def compute_prior_bugs(df_subset):
    df_subset = df_subset.sort_values('committed_at').copy()
    df_subset['prior_bugs_author'] = (
        df_subset.groupby('author')[TARGET]
        .transform(lambda x: x.shift(1).expanding().sum().fillna(0).astype(int))
        if 'author' in df_subset.columns
        else 0
    )
    return df_subset

df_train = compute_prior_bugs(df_train)

# For test: use the last known prior_bugs_author from train as seed,
# then roll forward within the test period only
if 'author' in df_test.columns:
    train_tail = df_train.groupby('author')[TARGET].sum().rename('author_bug_seed')
    df_test = df_test.join(train_tail, on='author')
    df_test['author_bug_seed'] = df_test['author_bug_seed'].fillna(0)
    df_test = df_test.sort_values('committed_at')
    df_test['prior_bugs_author'] = (
        df_test.groupby('author')[TARGET]
        .transform(lambda x: x.shift(1).expanding().sum().fillna(0).astype(int))
        + df_test['author_bug_seed']
    ).astype(int)
    df_test = df_test.drop(columns=['author_bug_seed'])

print('✅ prior_bugs_author recomputed without data leakage')

## Step 6 — Feature Engineering (fold-safe)

All new features are computed from existing numeric columns only —
no target or metadata is used, so they are safe to apply to both splits.

In [ ]:
def engineer_features(df_in):
    """
    All engineered features derived purely from raw numeric columns.
    Safe to apply independently to train and test sets.
    """
    d = df_in.copy()

    # Already in SZZ output — keep as-is:
    #   churn_ratio, test_ratio, complexity_per_file
    # Recompute here defensively in case they're 0-filled
    d['churn_ratio']         = (d['lines_added'] / (d['lines_deleted'] + 1)).round(3)
    d['test_ratio']          = (d['test_files_changed'] / (d['files_changed'] + 1)).round(3)
    d['complexity_per_file'] = (d['avg_complexity'] / (d['files_changed'] + 1)).round(3)

    # New engineered features
    d['log_lines_added']     = np.log1p(d['lines_added'])       # tame outliers
    d['log_files_changed']   = np.log1p(d['files_changed'])
    d['method_density']      = (d['num_methods'] / (d['files_changed'] + 1)).round(3)
    d['large_commit']        = (d['lines_added'] > d['lines_added'].quantile(0.9)).astype(int)
    d['has_test_changes']    = (d['test_files_changed'] > 0).astype(int)
    d['lines_touched']       = d['lines_added'] + d['lines_deleted']
    d['is_micro_commit']     = (d['lines_touched'] <= 5).astype(int)

    return d

df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

# ── Final feature list for training ─────────────────────────────────────────
# Explicitly whitelisted — no metadata, no target, no leaky columns
FEATURES = [
    # Raw SZZ features
    'lines_added', 'lines_deleted', 'files_changed',
    'test_files_changed', 'num_methods', 'avg_complexity',
    'commit_hour', 'day_of_week', 'is_weekend', 'is_night_commit',
    'prior_bugs_author',
    # SZZ-computed ratios
    'churn_ratio', 'test_ratio', 'complexity_per_file',
    # Newly engineered
    'log_lines_added', 'log_files_changed', 'method_density',
    'large_commit', 'has_test_changes', 'lines_touched', 'is_micro_commit',
]

# Filter to only columns that actually exist
FEATURES = [f for f in FEATURES if f in df_train.columns]

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[FEATURES]
y_test  = df_test[TARGET]

# Confidence-based sample weights (SZZ labels = 1.0, keyword = 0.4)
sample_weights_train = df_train['confidence'].fillna(1.0).values

print(f'Feature count : {len(FEATURES)}')
print(f'Features      : {FEATURES}')
print(f'Train X shape : {X_train.shape}')
print(f'Test  X shape : {X_test.shape}')

## Step 7 — Preprocessing Pipeline

In [ ]:
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

# ── SMOTE: oversample minority class on training data only ──────────────────
bug_count   = int(y_train.sum())
clean_count = int((y_train == 0).sum())
print(f'Before SMOTE — Clean: {clean_count}  Buggy: {bug_count}  Ratio: 1:{clean_count//max(bug_count,1)}')

if bug_count >= 5:   # SMOTE needs at least k_neighbors=5 minority samples
    k = min(5, bug_count - 1)
    smote = SMOTE(random_state=42, k_neighbors=k)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    # SMOTE creates synthetic points; weights are uniform after resampling
    sample_weights_res = None
    print(f'After  SMOTE — Clean: {(y_train_res==0).sum()}  Buggy: {(y_train_res==1).sum()}')
else:
    print(f'⚠️  Too few bug samples for SMOTE ({bug_count}). Using class_weight="balanced" instead.')
    X_train_res, y_train_res = X_train_scaled, y_train
    sample_weights_res = sample_weights_train

print('\n✅ Preprocessing complete')

## Step 8 — Train Models with TimeSeriesSplit CV

`TimeSeriesSplit` respects chronological order: each fold trains on earlier
commits and validates on later ones — the same as production usage.

In [ ]:
# TimeSeriesSplit with 5 folds
tscv = TimeSeriesSplit(n_splits=5)

# --- Model 1: Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_res, y_train_res)

# CV on original (pre-SMOTE) training data to get unbiased estimate
rf_cv_auc = cross_val_score(
    rf_model, X_train_scaled, y_train,
    cv=tscv, scoring='roc_auc'
)
rf_cv_ap = cross_val_score(
    rf_model, X_train_scaled, y_train,
    cv=tscv, scoring='average_precision'
)
print(f'Random Forest — CV ROC-AUC : {rf_cv_auc.mean():.4f} ± {rf_cv_auc.std():.4f}')
print(f'Random Forest — CV Avg-Prec: {rf_cv_ap.mean():.4f} ± {rf_cv_ap.std():.4f}')

In [ ]:
# --- Model 2: XGBoost ---
# scale_pos_weight computed on post-SMOTE data since that's what we fit on
spw = float((y_train_res == 0).sum()) / float((y_train_res == 1).sum())

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False,
)

xgb_cv_auc = cross_val_score(
    xgb_model, X_train_scaled, y_train,
    cv=tscv, scoring='roc_auc'
)
xgb_cv_ap = cross_val_score(
    xgb_model, X_train_scaled, y_train,
    cv=tscv, scoring='average_precision'
)
print(f'XGBoost       — CV ROC-AUC : {xgb_cv_auc.mean():.4f} ± {xgb_cv_auc.std():.4f}')
print(f'XGBoost       — CV Avg-Prec: {xgb_cv_ap.mean():.4f} ± {xgb_cv_ap.std():.4f}')

# Pick best model by AUC
best_model = rf_model  if rf_cv_auc.mean()  >= xgb_cv_auc.mean()  else xgb_model
best_name  = 'Random Forest' if rf_cv_auc.mean() >= xgb_cv_auc.mean() else 'XGBoost'
print(f'\n🏆 Best model selected: {best_name}')

## Step 9 — Evaluate on Held-Out Test Set

In [ ]:
y_pred      = best_model.predict(X_test_scaled)
y_pred_prob = best_model.predict_proba(X_test_scaled)[:, 1]

print(f'=== {best_name} — Test Set Evaluation ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC       : {roc_auc_score(y_test, y_pred_prob):.4f}')
print(f'Avg Precision : {average_precision_score(y_test, y_pred_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Clean', 'Buggy']))

print('=== Interpretation guide ===')
print('  ROC-AUC > 0.75 → good discrimination between clean/buggy')
print('  Avg Precision  → more informative than AUC for imbalanced data')
print('  High Recall    → catches more real bugs (fewer missed)')
print('  High Precision → fewer false alarms')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Clean', 'Buggy'],
            yticklabels=['Clean', 'Buggy'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC curve — both models
for model, name, color in [
    (rf_model,  'Random Forest', '#1976D2'),
    (xgb_model, 'XGBoost',       '#E53935'),
]:
    probs = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Precision-Recall curve (better metric for imbalanced data)
for model, name, color in [
    (rf_model,  'Random Forest', '#1976D2'),
    (xgb_model, 'XGBoost',       '#E53935'),
]:
    probs = model.predict_proba(X_test_scaled)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    axes[2].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, lw=2)
baseline = y_test.mean()
axes[2].axhline(baseline, linestyle='--', color='gray', label=f'Baseline ({baseline:.2f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.suptitle(f'{best_name} — Test Set', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(best_model.feature_importances_, index=FEATURES)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color='steelblue')
plt.title(f'Top Feature Importances — {best_name}')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 10 bug predictors:')
for feat, imp in top_features.head(10).items():
    print(f'  {feat:<30} {imp:.4f}')

In [ ]:
# Threshold analysis — find optimal threshold for your use case
# Bug prediction often prefers higher recall (catch more bugs)
# at the cost of more false positives
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_test, y_pred_prob)

plt.figure(figsize=(10, 4))
plt.plot(thresh_arr, prec_arr[:-1], label='Precision', color='#1976D2', lw=2)
plt.plot(thresh_arr, rec_arr[:-1],  label='Recall',    color='#E53935', lw=2)
f1 = 2 * prec_arr[:-1] * rec_arr[:-1] / (prec_arr[:-1] + rec_arr[:-1] + 1e-9)
plt.plot(thresh_arr, f1, label='F1', color='#388E3C', lw=2, linestyle='--')
best_t = thresh_arr[np.argmax(f1)]
plt.axvline(best_t, color='gray', linestyle=':', label=f'Best F1 threshold={best_t:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Threshold vs Precision / Recall / F1')
plt.legend()
plt.tight_layout()
plt.show()

OPTIMAL_THRESHOLD = round(float(best_t), 3)
print(f'Optimal threshold (max F1): {OPTIMAL_THRESHOLD}')
print('→ Use this in predict_bug_risk() below instead of 0.5')

## Step 10 — Save & Reload Model

In [ ]:
MODEL_PATH     = 'bug_predictor_model_v2.pkl'
PREP_PATH      = 'bug_predictor_preprocessor_v2.pkl'
FEATURES_PATH  = 'bug_predictor_features_v2.pkl'

joblib.dump(best_model,   MODEL_PATH)
joblib.dump(preprocessor, PREP_PATH)
joblib.dump(FEATURES,     FEATURES_PATH)

print(f'✅ Model saved       → {MODEL_PATH}')
print(f'✅ Preprocessor saved → {PREP_PATH}')
print(f'✅ Feature list saved → {FEATURES_PATH}')

# Reload & sanity check
loaded_model    = joblib.load(MODEL_PATH)
loaded_prep     = joblib.load(PREP_PATH)
loaded_features = joblib.load(FEATURES_PATH)

sanity_pred = loaded_model.predict(loaded_prep.transform(X_test[loaded_features]))
assert (sanity_pred == y_pred).all(), '❌ Reload sanity check FAILED'
print('✅ Sanity check passed — reloaded model predictions match!')

## Step 11 — Predict on New Commit

Pass a dict of raw commit metrics.  
The same feature engineering pipeline is applied automatically.

In [ ]:
def predict_bug_risk(
    commit_dict: dict,
    model,
    prep,
    feature_names: list,
    threshold: float = OPTIMAL_THRESHOLD,
) -> dict:
    """
    Predict bug probability for a single commit.

    commit_dict   : dict of raw feature values (same names as SZZ output)
    threshold     : decision threshold (default: optimal F1 threshold from Step 9d)
    Returns       : dict with prediction, probability, and risk level
    """
    df_in = pd.DataFrame([commit_dict])

    # Defaults for optional fields
    df_in.setdefault('lines_deleted',      0)
    df_in.setdefault('test_files_changed', 0)
    df_in.setdefault('num_methods',        0)
    df_in.setdefault('avg_complexity',     1)
    df_in.setdefault('prior_bugs_author',  0)
    df_in.setdefault('is_night_commit',    0)
    df_in.setdefault('is_weekend',         0)
    df_in.setdefault('day_of_week',        0)
    df_in.setdefault('commit_hour',        12)

    # Apply the same feature engineering
    df_in = engineer_features(df_in)

    # Ensure all required columns exist
    for col in feature_names:
        if col not in df_in.columns:
            df_in[col] = 0
    df_in = df_in[feature_names]

    X_scaled     = prep.transform(df_in)
    probability  = model.predict_proba(X_scaled)[0][1]
    is_buggy     = probability >= threshold

    risk_level = (
        '🟢 LOW'    if probability < 0.35 else
        '🟡 MEDIUM' if probability < 0.60 else
        '🔴 HIGH'
    )

    return {
        'is_buggy':    bool(is_buggy),
        'probability': round(float(probability), 4),
        'risk_level':  risk_level,
        'threshold':   threshold,
    }


# ── Example 1: risky commit ───────────────────────────────────────────────────
risky_commit = {
    'lines_added':        480,
    'lines_deleted':       20,
    'files_changed':       15,
    'test_files_changed':   0,
    'num_methods':          2,
    'avg_complexity':      45,
    'commit_hour':         23,
    'day_of_week':          6,
    'is_weekend':           1,
    'is_night_commit':      1,
    'prior_bugs_author':   40,
}

# ── Example 2: safe commit ───────────────────────────────────────────────────
safe_commit = {
    'lines_added':        30,
    'lines_deleted':        5,
    'files_changed':        1,
    'test_files_changed':   2,
    'num_methods':          3,
    'avg_complexity':       4,
    'commit_hour':         11,
    'day_of_week':          2,
    'is_weekend':           0,
    'is_night_commit':      0,
    'prior_bugs_author':    2,
}

for label, commit in [('Risky Commit', risky_commit), ('Safe Commit', safe_commit)]:
    r = predict_bug_risk(commit, loaded_model, loaded_prep, loaded_features)
    print(f'=== {label} ===')
    print(f'  Bug Predicted : {r["is_buggy"]}')
    print(f'  Probability   : {r["probability"]:.1%}')
    print(f'  Risk Level    : {r["risk_level"]}')
    print(f'  Threshold used: {r["threshold"]}')
    print()

---
## Summary of Changes vs v1

| Issue | v1 | v2 (this notebook) |
|---|---|---|
| Data source | `generate_dataset()` (synthetic) | `szz_final_labels.csv` (real commits) |
| Feature columns | Mismatched (review/experience features) | Aligned to SZZ output exactly |
| Train/test split | Random 80/20 | Temporal 80/20 by `committed_at` |
| `prior_bugs_author` | Computed on full dataset (leaky) | Recomputed per split (fold-safe) |
| Cross-validation | Random `StratifiedKFold` | `TimeSeriesSplit` (respects time order) |
| Class imbalance | `class_weight='balanced'` only | SMOTE on training fold + `class_weight` |
| Label confidence | Ignored | Used as sample weights & confidence filter |
| Evaluation metrics | Accuracy + ROC-AUC | + Avg Precision + PR curve + threshold sweep |
| Prediction threshold | Hard-coded 0.5 | Optimal F1 threshold from PR curve |
| Leaky features | `commit_type` risk (borderline) | Explicitly excluded from `FEATURES` list |

**To run end-to-end:**
1. Run `szz_v3_stable.ipynb` → produces `szz_final_labels.csv`
2. Run this notebook — it reads that CSV directly
3. Use `predict_bug_risk()` in Step 11 with any new commit's metrics